In [7]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [10]:
import importlib

from test_data.spam import spam_profile_descriptions
from test_data.gibberish import gibberish_profile_descriptions
from test_data.inappropriate import inappropriate_profile_descriptions
from test_data.unsupported_languages import unsupported_language_profile_descriptions
import moderate

importlib.reload(moderate)

agent = moderate.initialize_agent()

# select categories to evaluate
categories = [
    "spam",
    # "gibberish",
    # "inappropriate",
    # "language",
]

test_data = []

for category in categories:
    if category == "spam":
        for description in spam_profile_descriptions: #  [0:2]:
            test_data.append((description, "is_spam"))
    elif category == "gibberish":
        for description in gibberish_profile_descriptions: # [0:2]:
            test_data.append((description, "is_gibberish"))
    elif category == "inappropriate":
        for description in inappropriate_profile_descriptions: #[0:2]:
            test_data.append((description, "is_inappropriate"))
    elif category == "language":
        for description in unsupported_language_profile_descriptions: #[0:2]:
            test_data.append((description, "language"))

count_correct = 0
count_total = 0

incorrectly_classified = []

for profile, flag in test_data:
    result = moderate.moderate_profile_description(profile, agent)
    print(f"Ad: {profile}")
    print(f"Result: {result}")
    
    count_total += 1

    if flag != "language":
        if result[flag] is True:
            count_correct += 1
            print(f"Correctly evaluated as {flag}: {count_correct} out of {count_total}")
        else:
            incorrectly_classified.append((profile, result))
    else:
        if result["language"] not in moderate.accepted_languages:
            count_correct += 1
            print(f"Correctly evaluated with language {result['language']}: {count_correct} out of {count_total}")
        else:
            incorrectly_classified.append((profile, result))

    print("---")

print(f"Total correctly evaluated: {count_correct} out of {count_total}")

for profile, result in incorrectly_classified:
    print(f"Incorrectly classified: {profile}")
    print(f"Result: {result}")
    print("---")

Using model: gpt-5.4-nano
Ad: Castigi 5000 EUR pe saptamana fara munca! Scrie-mi pe WhatsApp pentru metoda secreta si intra azi in echipa castigatoare.
Result: {'is_valid': False, 'is_gibberish': False, 'is_spam': True, 'is_inappropriate': False, 'language': 'romanian', 'confidence': 0.93, 'reason': 'promovare scam și îndemn la WhatsApp'}
Correctly evaluated as is_spam: 1 out of 1
---
Ad: Nu mai cauta mesteri aici, intra pe linkul meu si primesti reduceri garantate la orice lucrare. Oferta expira in 2 ore!
Result: {'is_valid': False, 'is_gibberish': False, 'is_spam': True, 'is_inappropriate': False, 'language': 'romanian', 'confidence': 0.93, 'reason': 'Promoveaza link extern si reduceri cu urgenta'}
Correctly evaluated as is_spam: 2 out of 2
---
Ad: Promovam orice afacere instant pe Google si Facebook. Trimite mesaj privat pentru pachet VIP cu rezultate garantate si clienti nelimitati.
Result: {'is_valid': False, 'is_gibberish': False, 'is_spam': True, 'is_inappropriate': False, 'lang